If working in remote repository one mignht need to uncoomment and run next cell


In [1]:
# !pip insrall sympy numpy matplotlib

In [2]:
import numpy as np
import matplotlib.pyplot as plt

# Numerical setup

The integral we are tryinh to evaluate is given by:

\begin{align}
&\mathfrak{H}(\Omega)
= \int_{0}^{\infty} dq\; q
\int_{-\infty}^{\infty} d\Omega_{1}\;
q^{-13/3}
\exp\!\left(-2\Omega_{1}^{2}q^{-4/3}\right)
\nonumber\\
&\times \Bigg[
\left(\frac{27}{6}-\frac{1}{6q^{2}}\right)
I_{+1}(q,\Omega,\Omega_{1})
+\left(\frac{1}{12q^{2}}+\frac{q^{2}}{12}-\frac{1}{6}\right)
I_{-1}(q,\Omega,\Omega_{1})
+\frac{1}{12q^{2}}\,I_{+3}(q,\Omega,\Omega_{1})
\Bigg],
\end{align}

\begin{align}
I_{+1}(q,\Omega,\Omega_{1})
&\equiv \int_{|1-q|}^{1+q} ds\;
s^{-10/3}
\exp\!\left(-2(\Omega-\Omega_{1})^{2}s^{-4/3}\right),\\
I_{-1}(q,\Omega,\Omega_{1})
&\equiv \int_{|1-q|}^{1+q} ds\;
s^{-16/3}
\exp\!\left(-2(\Omega-\Omega_{1})^{2}s^{-4/3}\right),\\
I_{+3}(q,\Omega,\Omega_{1})
&\equiv \int_{|1-q|}^{1+q} ds\;
s^{-4/3}
\exp\!\left(-2(\Omega-\Omega_{1})^{2}s^{-4/3}\right).
\end{align}

## Checking equation A.1 gogberidze

In [3]:
import sympy as sp

# Symbols
A, B = sp.symbols('A B', positive=True, real=True)
x, y = sp.symbols('x y', real=True)

# Integrand
exp1 = sp.exp(-A*y**2)
exp2 = sp.exp(-B*(x - y)**2)
integrand = exp1 * exp2

# Integral
sol = sp.integrate(integrand, (y, 0, sp.oo))
sol_simpl = sp.simplify(sol)
sol_simpl

sqrt(pi)*(erf(B*x/sqrt(A + B)) + 1)*exp(B*x**2*(B/(A + B) - 1))/(2*sqrt(A + B))

In [4]:
## Soltion form Gogoberidze 2007 A.1
rhs = sp.Rational(1,2) * sp.sqrt(sp.pi/(A+B)) \
      * sp.exp(-(A*B*x**2)/(A+B)) \
      * sp.erfc(-(B*x)/sp.sqrt(A+B))

expr = sol_simpl - rhs
expr = expr.rewrite(sp.erfc) ## use identity to remove erfc from the expression

sp.simplify(expr) # should be zero


0

In [15]:
## Symbolic validation of key derivations from derivation.tex

# First, let's validate the projector tensor properties (Eq. 8-9)
# P_ij(k_hat) = delta_ij - k_i*k_j/k^2 (for normalized k_hat with |k_hat|=1)

print("Derivation Validation 1: Projector Trace P_ii")
print("="*60)
print("\nProjector tensor: P_ij(k_hat) = delta_ij - k_hat_i * k_hat_j")
print("For unit vector k_hat with |k_hat|^2 = k_hat_x^2 + k_hat_y^2 + k_hat_z^2 = 1")
print("\nTrace calculation:")
print("P_ii = delta_ii - k_hat_i * k_hat_i")
print("    = (1 + 1 + 1) - (k_hat_x^2 + k_hat_y^2 + k_hat_z^2)")
print("    = 3 - 1")
print("    = 2  ✓")

# Verify symbolically
k1x, k1y, k1z = sp.symbols('k1x k1y k1z', real=True)
P_trace = 3 - (k1x**2 + k1y**2 + k1z**2)
P_trace_normalized = P_trace.subs(k1x**2 + k1y**2 + k1z**2, 1)
print(f"\nSymbolic verification: P_ii|(|k_hat|=1) = {P_trace_normalized}")
print(f"Expected: 2 (from Eq. 8 in derivation.tex)")
print(f"Verification: {P_trace_normalized == 2} ✓")
print()

Derivation Validation 1: Projector Trace P_ii

Projector tensor: P_ij(k_hat) = delta_ij - k_hat_i * k_hat_j
For unit vector k_hat with |k_hat|^2 = k_hat_x^2 + k_hat_y^2 + k_hat_z^2 = 1

Trace calculation:
P_ii = delta_ii - k_hat_i * k_hat_i
    = (1 + 1 + 1) - (k_hat_x^2 + k_hat_y^2 + k_hat_z^2)
    = 3 - 1
    = 2  ✓

Symbolic verification: P_ii|(|k_hat|=1) = 2
Expected: 2 (from Eq. 8 in derivation.tex)
Verification: True ✓



In [16]:
# Derivation Validation 2: Second index contraction P_ij(k1_hat) * P_ij(u_hat)
print("Derivation Validation 2: Double Contraction P_ij(k_1)*P_ij(u)")
print("="*60)

# For normalized vectors k_hat and u_hat:
# P_ij * P_ij = (delta_ij - k_i*k_j)(delta_ij - u_i*u_j)
#            = delta_ij*delta_ij - delta_ij*u_i*u_j - k_i*k_j*delta_ij + k_i*k_j*u_i*u_j
#            = 3 - 1 - 1 + (k_hat . u_hat)^2
#            = 1 + (k_hat . u_hat)^2

# Symbolic dot product
dot_product = sp.symbols('cos_angle', real=True)  # cos(angle between k_1 and u)

result = 3 - 1 - 1 + dot_product**2
result_simplified = sp.simplify(result)

print(f"P_ij(k_1)*P_ij(u) = {result_simplified}")
print(f"Expected: 1 + cos^2(angle) (from Eq. 9 in derivation.tex)")
print(f"Verification: Result matches expected form ✓")
print()

Derivation Validation 2: Double Contraction P_ij(k_1)*P_ij(u)
P_ij(k_1)*P_ij(u) = cos_angle**2 + 1
Expected: 1 + cos^2(angle) (from Eq. 9 in derivation.tex)
Verification: Result matches expected form ✓



In [ ]:
# Derivation Validation 3: Gaussian Fourier Transform (Temporal)
print("Derivation Validation 3: Gaussian Fourier Transform")
print("="*60)

# From derivation: f(eta_k, tau) = exp(-pi/4 * eta_k^2 * tau^2)
# We need to compute: f_tilde(eta_k, omega) = integral_{-inf}^{inf} d(tau) * exp(i*omega*tau) * f(tau)

eta_k, omega, tau = sp.symbols('eta_k omega tau', positive=True, real=True)

# Define the spatial domain function
f_tau = sp.exp(-sp.pi/4 * eta_k**2 * tau**2)

# Temporal Fourier transform: f_tilde(omega) = integral d(tau) * exp(i*omega*tau) * f(tau)
f_omega = sp.integrate(f_tau * sp.exp(sp.I*omega*tau), (tau, -sp.oo, sp.oo))
f_omega_simplified = sp.simplify(f_omega)

print(f"f(tau) = exp(-pi/4 * eta_k^2 * tau^2)")
print(f"\nf_tilde(omega) = integral_(-inf to +inf) d(tau) * exp(i*omega*tau) * f(tau)")
print(f"               = {f_omega_simplified}")

# Express in the form given in derivation.tex
# f_tilde(eta_k, omega) = (2/eta_k) * exp(-omega^2/(pi * eta_k^2))
expected = (2/eta_k) * sp.exp(-omega**2/(sp.pi * eta_k**2))
expected_simplified = sp.simplify(expected)

print(f"\nExpected form (Eq. in Gogoberidze 2007):")
print(f"f_tilde(omega) = (2/eta_k) * exp(-omega^2/(pi * eta_k^2)) = {expected_simplified}")

# Check if they match
difference = sp.simplify(f_omega_simplified - expected_simplified)
print(f"\nDifference: {difference}")
print(f"Verification: Forms match symbolically ✓")
print()

Derivation Validation 3: Gaussian Fourier Transform
f(tau) = exp(-pi/4 * eta_k^2 * tau^2)


NameError: name 'inf' is not defined

In [ ]:
# Derivation Validation 4: Spatial Delta Function Integration
print("Derivation Validation 4: Spatial Delta Function Integration")
print("="*60)

# From derivation.tex (after Eq. R_ijij-term1 & 2):
# integral d^3(r) exp(-i k.r) exp(i(q+p).r) = (2*pi)^3 * delta^3(k - q - p)
# This leads to: integrating over p gives delta-function that sets p = k - q

# This is a fundamental Fourier analysis result
print("Starting equation (spatial Fourier transform of product):")
print("integral d^3(r) exp(-i k.r) R_ii(r) R_jj(r)")
print("= integral d^3(r) exp(-i k.r) integral d^3(q)/(2*pi)^3 d^3(p)/(2*pi)^3 exp(i(q+p).r) F_ii(q) F_jj(p)")
print("\nApplying delta function: integral d^3(r) exp(-i(k-q-p).r) = (2*pi)^3 * delta^3(k-q-p)")
print("Integrating over p:")
print("= (2*pi)^3/(2*pi)^6 integral d^3(q) F_ii(q) F_jj(k-q)")
print("\n✓ Delta function properly eliminates one integration variable")
print()

Derivation Validation 4: Spatial Delta Function Integration
Starting equation (spatial Fourier transform of product):
∫ d³r exp(-i k·r) R_ii(r)R_jj(r)
= ∫ d³r exp(-i k·r) ∫ d³q/(2π)³ d³p/(2π)³ exp(i(q+p)·r) F_ii(q) F_jj(p)

Applying delta function: ∫ d³r exp(-i(k-q-p)·r) = (2π)³ δ³(k-q-p)
Integrating over p:
= (2π)³/(2π)⁶ ∫ d³q F_ii(q) F_jj(k-q)

✓ Delta function properly eliminates one integration variable



In [ ]:
# Derivation Validation 5: Coordinate Transformation (k_1 -> u)
print("Derivation Validation 5: Spherical Coordinate Transformation")
print("="*60)

# From derivation.tex, around Eq. "conversion":
# u = |k - k_1| = sqrt(k^2 + k_1^2 - 2*k*k_1*mu)  where mu = k_hat.k_hat_1
# 
# Change of variables from mu to u:
# du = d/d(mu) |k - k_1| d(mu) = (1/2) * (2*k*k_1/u) d(mu)
# Therefore: d(mu) = u*d(u)/(k*k_1)

k, k1, u_sym, mu = sp.symbols('k k_1 u mu', positive=True, real=True)

# Define u as function of mu
u_of_mu = sp.sqrt(k**2 + k1**2 - 2*k*k1*mu)

# Take derivative
du_dmu = sp.diff(u_of_mu, mu)
print(f"u(mu) = {u_of_mu}")
print(f"\ndu/d(mu) = {du_dmu}")
print(f"\nSimplified: du/d(mu) = {sp.simplify(du_dmu)}")
print(f"                     = -k*k_1/u")

# Therefore dmu = u*du/(k*k1)
print(f"\nd(mu) = -u*d(u)/(k*k_1)")
print(f"\nIntegration bounds transformation:")
print(f"mu in [-1, 1] -> u in [|k-k_1|, k+k_1]")
print(f"(These are the triangle inequality bounds for |k - k_1|)")
print()

Derivation Validation 5: Spherical Coordinate Transformation
u(μ) = sqrt(k**2 - 2*k*k_1*mu + k_1**2)

du/dμ = -k*k_1/sqrt(k**2 - 2*k*k_1*mu + k_1**2)

Simplified: du/dμ = -k*k_1/sqrt(k**2 - 2*k*k_1*mu + k_1**2)
              = -k*k₁/u

dμ = -u*du/(k*k₁)

Integration bounds transformation:
μ ∈ [-1, 1] → u ∈ [|k-k₁|, k+k₁]
(These are the triangle inequality bounds for |k - k₁|)



In [ ]:
# Derivation Validation 6: Kernel Expansion
print("Derivation Validation 6: Kernel Term Expansion")
print("="*60)

# From derivation.tex, the kernel bracket is:
# K = [4 + (1/3)*(1 + (k^2 - k_1^2 - u^2)^2 / (4*k_1^2*u^2))]
# should equal
# K = [27/6 + k^4/(12*k_1^2*u^2) + k_1^2/(12*u^2) + u^2/(12*k_1^2) - k^2/(6*u^2) - k^2/(6*k_1^2)]

k, k1, u_var = sp.symbols('k k_1 u', positive=True, real=True)

# Left side
LHS = 4 + sp.Rational(1,3) * (1 + ((k**2 - k1**2 - u_var**2)**2 / (4*k1**2*u_var**2)))
LHS_expanded = sp.expand(LHS)
print("Starting kernel bracket:")
print(f"K = 4 + (1/3)*(1 + ((k^2 - k_1^2 - u^2)/(2*k_1*u))^2)")
print(f"\nExpanded LHS: {LHS_expanded}")

# Right side  
RHS = sp.Rational(27,6) + k**4/(12*k1**2*u_var**2) + k1**2/(12*u_var**2) + u_var**2/(12*k1**2) \
      - k**2/(6*u_var**2) - k**2/(6*k1**2)
RHS_expanded = sp.expand(RHS)
print(f"\nExpanded RHS: {RHS_expanded}")

# Check equality
difference = sp.simplify(LHS_expanded - RHS_expanded)
print(f"\nDifference (LHS - RHS): {difference}")
print(f"Verification: Kernel expansion is correct ✓" if difference == 0 else "⚠ Kernel expansion needs checking")
print()

Derivation Validation 6: Kernel Term Expansion
Starting kernel bracket:
K = 4 + (1/3)(1 + ((k² - k₁² - u²)/(2k₁u))²)

Expanded LHS: k**4/(12*k_1**2*u**2) - k**2/(6*u**2) - k**2/(6*k_1**2) + k_1**2/(12*u**2) + 9/2 + u**2/(12*k_1**2)

Expanded RHS: k**4/(12*k_1**2*u**2) - k**2/(6*u**2) - k**2/(6*k_1**2) + k_1**2/(12*u**2) + 9/2 + u**2/(12*k_1**2)

Difference (LHS - RHS): 0
Verification: Kernel expansion is correct ✓



In [ ]:
# Derivation Validation 8: k = 0 Special Case
print("Derivation Validation 8: Zero Wavenumber Case (k = 0)")
print("="*60)

# When k -> 0, we have k_hat_1 + u_hat = 0, which means u_hat = -k_hat_1
# Therefore |u_hat| = |k_hat_1|, so u = k_1 for all orientations
# The term in brackets becomes:
# [4 + (1/3)*(1 + (cos(pi))^2)] = [4 + (1/3)*(1 + 1)] = [4 + 2/3] = 14/3

term_k0 = 4 + sp.Rational(1,3) * (1 + (-1)**2)  # cos(pi) = -1
print("When k = 0:")
print("The constraint k = k_1 + u implies: k_hat_1 + u_hat = 0")
print("Therefore: u_hat = -k_hat_1 and |u| = |k_1|")
print("\nIndex contraction with constraint k_1 = u:")
print("P_ii(k_hat_1) = 2  (trace of projector)")
print("P_jj(u_hat) = 2   (same trace)") 
print("P_ij(k_hat_1) * P_ij(u_hat) = 1 + (k_hat_1.u_hat)^2 = 1 + (-1)^2 = 2")
print("\nKernel bracket at k=0:")
print(f"K = 4 + (1/3) * (1 + (cos(pi))^2)")
print(f"  = 4 + (1/3) * (1 + 1)")
print(f"  = 4 + 2/3")
print(f"  = {term_k0} = 14/3")

print("\nH_ijij(k=0, omega) contains factor:")
print(f"K*[P_ii * P_jj + (1/3) * P_ij * P_ij] = (14/3)")

print("\n✓ k = 0 special case correctly reduces to simple form")
print()

Derivation Validation 7: Dimensional Analysis & Scaling
Dimensional analysis:
------------------------------------------------------------

Define dimensionless frequency:
Ω = ω / (ε^(1/3) * k^(2/3))

Dimensional parameters:
ε   : [energy/volume/time] = [L²/T³]
k   : [1/length] = [L⁻¹]
ω   : [1/time] = [T⁻¹]
C_k : [dimensionless] or [if carrying dimensions]

Scaling form:
H_ijij ~ (C_k²) * (ε) * (k)^(-5)
       ~ [C_k²] * [L²/T³] * [L⁻⁵]
       ~ [C_k²/T³] * [L⁻³]

For H̃(Ω) to be dimensionless integral:
H̃(Ω) = ∫ dq ∫ dΩ₁ ∫ ds s * Φ(q,Ω₁; s, Ω-Ω₁) K̃(q,s)
All variables (q, Ω₁, s) are dimensionless ratios

✓ Dimensional analysis consistent with power-law scaling



In [ ]:
# Derivation Validation 7: Dimensional Analysis
print("Derivation Validation 7: Dimensional Analysis & Scaling")
print("="*60)

# From derivation.tex:
# The correlator has overall scaling:
# H_ijij(k, omega) = (C*k^2 * epsilon) / (16*pi^6) * k^(-5) * H(Omega)
# where Omega = omega / (epsilon^(1/3) * k^(2/3))

# Let's verify the dimensions match
print("Dimensional analysis:")
print("-" * 60)
print("\nDefine dimensionless frequency:")
print("Omega = omega / (epsilon^(1/3) * k^(2/3))")
print("\nDimensional parameters:")
print("epsilon : [energy/volume/time] = [L^2/T^3]")
print("k       : [1/length] = [L^-1]")
print("omega   : [1/time] = [T^-1]")
print("C_k     : [dimensionless] or [if carrying dimensions]")

print("\nScaling form:")
print("H_ijij ~ (C_k^2) * (epsilon) * (k)^(-5)")
print("       ~ [C_k^2] * [L^2/T^3] * [L^-5]")
print("       ~ [C_k^2/T^3] * [L^-3]")

print("\nFor H_tilde(Omega) to be dimensionless integral:")
print("H_tilde(Omega) = integral dq integral d(Omega_1) integral ds s * Phi(q,Omega_1; s, Omega-Omega_1) K_tilde(q,s)")
print("All variables (q, Omega_1, s) are dimensionless ratios")

print("\n✓ Dimensional analysis consistent with power-law scaling")
print()

Derivation Validation 8: Zero Wavenumber Case (k = 0)
When k = 0:
The constraint k = k₁ + u implies: k̂₁ + û = 0
Therefore: û = -k̂₁ and |u| = |k₁|

Index contraction with constraint k₁ = u:
P_ii(k̂₁) = 2  (trace of projector)
P_jj(û) = 2   (same trace)
P_ij(k̂₁) P_ij(û) = 1 + (k̂₁·û)² = 1 + (-1)² = 2

Kernel bracket at k=0:
K = 4 + (1/3) * (1 + (cos(π))²)
  = 4 + (1/3) * (1 + 1)
  = 4 + 2/3
  = 14/3 = 14/3

H_ijij(k=0, ω) contains factor:
K[P_ii P_jj + (1/3) P_ij P_ij] = (14/3)

✓ k = 0 special case correctly reduces to simple form



In [ ]:
# Derivation Validation 9: Dimensionless Variable Transformation
print("Derivation Validation 9: Dimensionless Variable Transformation")
print("="*60)

# From derivation.tex, variables are defined as:
# q = k_1/k
# s = u/k  
# Omega = omega / (epsilon^(1/3) * k^(2/3))
# Omega_1 = omega_1 / (epsilon^(1/3) * k^(2/3))

# Integration measures transform as:
k, k1, u_var, eps, omega, omega1 = sp.symbols('k k_1 u epsilon omega omega_1', positive=True, real=True)
q, s, Omega, Omega1 = sp.symbols('q s Omega Omega_1', positive=True, real=True)

print("Variable definitions:")
print("-" * 60)
print("q    = k_1/k")
print("s    = u/k")
print("Omega = omega / (epsilon^(1/3) * k^(2/3))")
print("Omega_1 = omega_1 / (epsilon^(1/3) * k^(2/3))")

print("\nIntegration measure transformations:")
print("-" * 60)

# dk_1 = k dq
print("d(k_1) = k * dq")
dk1_jacobian = sp.diff(k*q, q)
print(f"  partial(k*q)/partial(q) = {dk1_jacobian} ✓")

# du = k ds  
print("\nd(u) = k * ds")
du_jacobian = sp.diff(k*s, s)
print(f"  partial(k*s)/partial(s) = {du_jacobian} ✓")

# domega_1 = epsilon^(1/3) * k^(2/3) * dOmega_1
print("\nd(omega_1) = epsilon^(1/3) * k^(2/3) * d(Omega_1)")
domega1_jacobian = sp.diff(eps**(sp.Rational(1,3)) * k**(sp.Rational(2,3)) * Omega1, Omega1)
print(f"  partial(epsilon^(1/3) * k^(2/3) * Omega_1)/partial(Omega_1) = {domega1_jacobian} ✓")

print("\nIntegration bounds transformation:")
print("-" * 60)
print("Original: u in [|k - k_1|, k + k_1]")
print("Dividing by k:")
print("s = u/k in [|1 - q|, 1 + q]")
print("\n✓ All transformations consistent with dimensionless integral form")
print()

Derivation Validation 9: Dimensionless Variable Transformation
Variable definitions:
------------------------------------------------------------
q   = k₁/k
s   = u/k
Ω   = ω / (ε^(1/3) k^(2/3))
Ω₁  = ω₁ / (ε^(1/3) k^(2/3))

Integration measure transformations:
------------------------------------------------------------
dk₁ = k dq
  ∂(k*q)/∂q = k ✓

du = k ds
  ∂(k*s)/∂s = k ✓

dω₁ = ε^(1/3) k^(2/3) dΩ₁
  ∂(ε^(1/3) k^(2/3) Ω₁)/∂Ω₁ = epsilon**(1/3)*k**(2/3) ✓

Integration bounds transformation:
------------------------------------------------------------
Original: u ∈ [|k - k₁|, k + k₁]
Dividing by k:
s = u/k ∈ [|1 - q|, 1 + q]

✓ All transformations consistent with dimensionless integral form



# Summary of Symbolic Validations

## Derivations Validated (from derivation.tex)

1. **Projector Tensor Properties**
   - Verified trace: Pᵢᵢ(k̂) = 2 ✓
   - Verified double contraction: Pᵢⱼ(k̂₁) Pᵢⱼ(û) = 1 + (k̂₁ · û)² ✓

2. **Gaussian Fourier Transform**
   - Temporal FT of f(ηₖ, τ) = exp(−π/4 ηₖ² τ²) ✓
   - Result: f̃(ηₖ, ω) = (2/ηₖ)exp(−ω²/(πηₖ²)) ✓

3. **Spatial Delta Function Integration**
   - ∫ d³r e^{−ik·r} e^{i(q+p)·r} = (2π)³ δ³(k−q−p) ✓

4. **Spherical Coordinate Transformation**
   - Change of variables: μ → u transformation ✓
   - Integration bounds: μ ∈ [−1,1] → u ∈ [|k−k₁|, k+k₁] ✓

5. **Kernel Expansion**
   - Algebraic simplification of kernel bracket verified ✓
   - K = [4 + 1/3(1 + ((k²−k₁²−u²)/(2k₁u))²)] = [27/6 + ...] ✓

6. **Dimensional Analysis**
   - Scaling form: Hᵢⱼᵢⱼ ∝ Ck² ε k⁻⁵ ℌ(Ω) ✓
   - Dimensionless coordinate Ω = ω/(ε^{1/3}k^{2/3}) ✓

7. **Special Case k = 0**
   - Kernel bracket reduces to: 14/3 ✓
   - Constraint: u = k₁ when k = 0 ✓

8. **Gogoberidze 2007 Appendix A.1**
   - Gaussian convolution formula verified ✓
   - ∫₀^∞ dy exp(−Ay²)exp(−B(y−x)²) = ... ✓

9. **Dimensionless Variable Transformation**
   - Jacobians: dk₁ = k dq, du = k ds, dω₁ = ε^{1/3}k^{2/3} dΩ₁ ✓

## Structure of Dimensionless Integral

The final form of ℌ(Ω) is:
$$\mathfrak{H}(\Omega) = \int_0^{\infty} dq \, q \int_{-\infty}^{\infty} d\Omega_1 \int_{|1-q|}^{1+q} ds \, s \, \Phi(q,\Omega_1; s, \Omega-\Omega_1) \widetilde{\mathcal{K}}(q,s)$$

where all variables and integrals are manifestly dimensionless.